# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** E Commerce Dataset.xlsx（使用 E Comm 工作表）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。

---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()

### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？

In [ ]:
df.info()

# 答案：
# 1.一行代表一个客户（用户）的完整画像信息，涵盖了该客户的基本属性、行为特征、偏好设置以及消费记录等20个维度的信息。例如 CustomerID=50001 的行记录了一位女性客户的全部信息：她使用 Mobile Phone 登录、居住在3线城市、距离仓库6公里、偏好借记卡支付、在App上花费3小时、注册了3台设备、偏好购买笔记本电脑及配件、满意度评分为2、单身、有9个地址、上月有投诉、订单金额同比增长11%、使用了1张优惠券、上月下单1次、距上次订单5天、获得返现159.93
# 2.CustomerID 是用户唯一标识。从示例数据可以看到 50001、50002、50003 等连续递增的编号，每个客户对应一个独立的 ID。
# 3.Churn 是流失分析的目标变量。该字段为二分类变量（示例中值为 0 或 1），其中 1 表示客户已流失，0 表示客户未流失。其他 19 个字段均可作为特征变量，用于构建流失预测模型。

---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。

### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。

In [ ]:
# TODO：创建 missing_report
# 提示：df.isna().sum() 统计缺失数量；df.isna().mean() 计算缺失比例。

missing_report = None

# display(missing_report)
# 创建 missing_report
missing_report = pd.DataFrame({
    "缺失数量": df.isna().sum(),
    "缺失比例": (df.isna().mean() * 100).round(2)
}).sort_values("缺失数量", ascending=False)

display(missing_report)

### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？

In [ ]:
# TODO：完成两项重复值统计
# duplicate_rows =df.duplicated().sum()
# duplicate_customer_ids =df['CustomerID'].duplicated().sum()

# print("完全重复行数：", duplicate_rows)
# print("CustomerID 重复数量：", duplicate_customer_ids)

# 思考：用户 ID 重复时，不能直接删除，因为……
# 完成两项重复值统计
# 1. 可能是同一用户在不同时间点的多次记录（如历史订单快照），直接删除会丢失时间维度信息；
# 2. 需要先查看重复行的具体内容，判断是数据录入错误还是合理重复；
# 3. 如果是数据错误，应保留最新/最完整的一条，而非盲目删除；
# 4. 直接删除可能导致样本量减少，影响后续模型训练效果。

---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。

### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。

In [ ]:
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

# TODO：循环填充每列的中位数
# for col in numeric_missing_cols:
#      df[col] = df[col].fillna(df[col].median())

# TODO：输出上述字段剩余的缺失值数量
# print(df[numeric_missing_cols].isna().sum())
| 字段                | 填充前缺失数 | 中位数  | 填充后缺失数 |
| --------------------------- | ------ | ---- | ------ |
| Tenure                      | 264    | 9.0  | 0      |
| WarehouseToHome             | 251    | 14.0 | 0      |
| HourSpendOnApp              | 255    | 3.0  | 0      |
| OrderAmountHikeFromlastYear | 265    | 15.0 | 0      |
| CouponUsed                  | 256    | 1.0  | 0      |
| OrderCount                  | 258    | 2.0  | 0      |
| DaySinceLastOrder           | 307    | 3.0  | 0      |


### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。

In [ ]:
# 场景：字段为分类型数据（如 PreferredPaymentMode 支付方式、Gender 性别），或字段缺失比例过高（如超过 50%），且缺失本身可能携带业务含义（如"用户未填写收入"可能暗示该用户不愿透露或收入特殊）。
# 处理思路：分类型数据：不能用中位数（中位数仅适用于数值型），应使用众数（Mode）填充，或单独设一个"未知"类别，保留缺失信息。
缺失比例高或缺失非随机：不宜简单填充，应分析缺失原因。若缺失与目标变量相关，可单独作为一列标记，再用模型预测填充，或根据业务规则分组填充，避免全局中位数扭曲数据分布。

---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。

### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。

In [ ]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: PreferredLoginDevice, dtype: int64

PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: PreferredPaymentMode, dtype: int64

PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: PreferedOrderCat, dtype: int64

### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。

In [ ]:
# TODO：完成类别标准化
# df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace("Phone", "Mobile Phone")
# df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({"COD": "Cash on Delivery", "CC": "Credit Card"})
# df["PreferedOrderCat"] = df["PreferedOrderCat"].replace("Mobile", "Mobile Phone")


# TODO：重新检查类别频数
# for col in category_cols:
#     print(f"\n{col}")
#     print(df[col].value_counts())
print("\n" + "=" * 60)
print("类别标准化")
print("=" * 60)

# 定义标准化映射规则
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({
    "Phone": "Mobile Phone"
})
df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
    "COD": "Cash on Delivery",
    "CC": "Credit Card"
})
df["PreferedOrderCat"] = df["PreferedOrderCat"].replace({
    "Mobile": "Mobile Phone"
})

# 重新检查类别频数
category_cols = ["PreferredLoginDevice", "PreferredPaymentMode", "PreferedOrderCat"]
for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())


---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。

In [ ]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })

### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？

In [ ]:
# TODO：调用函数检查三个字段
# display(iqr_outlier_summary(df["WarehouseToHome"]))
# display(iqr_outlier_summary(df["OrderCount"]))
# display(iqr_outlier_summary(df["CashbackAmount"]))

# 结论：候选异常值不能直接删除，因为业务真实性：统计上的"异常"不等于错误。例如 OrderCount 高达 16 的用户可能是平台的忠实高消费客户，CashbackAmount 为 0 可能只是该用户未参与返现活动，这些都是真实业务场景。
比例过高：OrderCount 异常值比例高达 12.49%，若直接删除将损失大量样本，严重改变数据分布，导致模型训练偏倚。
需结合业务判断：WarehouseToHome 为 127 虽远超正常范围，但可能是偏远地区用户；应先核实是否为录入错误，而非直接删除。
信息损失：异常值往往携带重要信息（如高价值用户识别），直接删除会削弱模型的区分能力。

### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

In [ ]:
# TODO：完成业务规则检查
# rules = {
#     "使用时长小于 0": (df["Tenure"] < 0).sum(),
#     "仓库距离小于 0": (df["WarehouseToHome"] < 0).sum(),
#     "订单数小于或等于 0": (df["OrderCount"] <= 0).sum(),
#     "返现金额小于 0": (df["CashbackAmount"] < 0).sum(),
# }
# pd.Series(rules)


---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。

In [ ]:
# TODO：完成验收
# assert df[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"
# assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
# assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
# assert "CC" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"

# print("数据清洗验收通过。")

### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。

In [ ]:
OUTPUT_PATH = Path("../output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# TODO：导出为 UTF-8-SIG 编码的 CSV 文件
# df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

# print(f"已导出：{OUTPUT_PATH.resolve()}")

---
## 7. 提交前自查

- [ ] 已完成缺失值报告；
- [ ] 已完成重复记录检查；
- [ ] 已填补指定数值字段的缺失值；
- [ ] 已统一登录设备、支付方式和订单品类；
- [ ] 已完成候选异常值与业务规则检查；
- [ ] 已导出 ecommerce_customer_cleaned.csv；
- [ ] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？